# Notebook 01 — Production Structure Pipeline

Notebook này có một pipeline production duy nhất: tự resolve/restore Phase00, restore checkpoint theo `video_id + stage`, xử lý tuần tự từng video, package và sync Phase01. Chỉ sửa cell `USER SETTINGS`, sau đó **Run All**.

Phase01 v1.4 dùng NVIDIA FastConformer cho ASR, gate ảnh không có chữ trước Vintern-1B-v3_5, rồi dùng một Qwen2.5-VL-7B-Instruct 4-bit xuyên caption, scene boundary và scene summary. Nếu Qwen lỗi, runtime unload Qwen trước khi chuyển cố định sang Vintern-3B-R-beta local cho phần còn lại của chunk.

Business logic và cấu hình deterministic nằm trong `src/system1/` và `configs/`. Local/Colab/Kaggle chỉ là scratch/runtime; checkpoint Hugging Face có quyền ghi (public hoặc private) là resume authority. Lưu ý: checkpoint public cũng làm các artifact trung gian có thể được đọc công khai.

Repo được lấy từ GitHub bằng `git clone` lần đầu và `git fetch` + fast-forward ở lần sau (`git pull --ff-only` tương đương). Package được cài bằng `pip install -e` với extra production. Các root tương thích orchestration gồm `AIC_REPO_ROOT`, `AIC_REPO_PARENT`, `AIC_DATA_ROOT`, `AIC_RUNTIME_ROOT`, và `AIC_ARTIFACT_ROOT`.

In [ ]:
# USER SETTINGS — teammate chỉ sửa các giá trị trong cell này, không sửa các cell pipeline bên dưới.
# Mỗi người nhận đúng batch do Notebook 00B tạo tại <release>/manifests/batch_000.txt, batch_001.txt, ...
# Không tự đặt batch_id mới: tên không khớp manifest sẽ làm worker không tìm thấy danh sách video.
batch_id = "batch_000"


# worker_id dùng cho report/log/checkpoint ownership. Các worker chạy song song phải dùng ID khác nhau.
# Khuyến nghị ghép cùng số với batch để dễ audit: batch_003 -> worker_003.
worker_id = "worker_000"


# Thường giữ None để chọn release Phase00 hoàn tất mới nhất theo completed_at.
# AIOU26_release hiện có completed_at nên giữ None để auto-resolve. Chỉ override khi cả team muốn pin một release cụ thể.
release_id_override = None

# HF STORAGE — giữ None để dùng versioned default trong configs/storage.yaml:
#   release: 1thesudden/AIOU26_release; checkpoint: 1thesudden/AIOU26_checkpoints.
# Chỉ đặt AIOU26_release_test khi repo test đã tồn tại và HF_TOKEN có quyền ghi. Không dùng tên AIC2026/AIC26 cũ.
hf_release_repo = None    # Dùng 1thesudden/AIOU26_release từ config; mọi worker trong cùng run phải trỏ cùng repo.
hf_checkpoint_repo = None    # State/artifact resume; public hoặc private, nhưng HF_TOKEN bắt buộc có quyền ghi.

# revision/prefix là scope nâng cao. Giữ None nếu không có kế hoạch namespace riêng cho cả team.
hf_revision = None
hf_prefix = None
checkpoint_revision = None
checkpoint_prefix = None  # Chỉ tách checkpoint TEST khi cả team dùng cùng namespace riêng.

# ASR: None hoặc "nemo" = NVIDIA FastConformer/Parakeet mặc định.
# Chỉ đặt "faster_whisper" để dùng Systran/faster-whisper-large-v3 thay thế.
asr_provider = None
# Scratch chứa video/model/artifact tạm và có thể rất lớn. None chọn /content hoặc /kaggle/temp an toàn hơn.
# Nếu override, dùng ổ local nhanh còn ít nhất 35 GiB; không trỏ vào Google Drive hoặc thư mục output được auto-save.
scratch_dir_override = None

# SOURCE CODE — branch thử nghiệm phải khớp code đang được validate trước khi tích hợp vào dev.
github_repo_url = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
github_branch = "fix-qwen"
repo_dir_name = "Multimodal-Agentic-Retrieval-Engine"

# Secrets không đặt trực tiếp trong notebook. HF_TOKEN hoặc AIC_HF_TOKEN bắt buộc để restore/sync.


## Cách teammate tự setting để chạy Notebook 01

Chỉ cần sửa cell `USER SETTINGS` phía trên, sau đó chọn **Run All**.

`batch_id` là batch mà máy/người này sẽ xử lý. Giá trị này phải khớp với file do Notebook 00B đã tạo trong `manifests/`, ví dụ `batch_000.txt`, `batch_001.txt`, ...

`worker_id` là tên worker dùng để ghi log/report. Nên đặt trùng số với batch cho dễ theo dõi.

Ví dụ nếu Notebook 00B tạo 10 batch:

```text
Người 1:  batch_id = "batch_000", worker_id = "worker_000"
Người 2:  batch_id = "batch_001", worker_id = "worker_001"
Người 3:  batch_id = "batch_002", worker_id = "worker_002"
...
Người 10: batch_id = "batch_009", worker_id = "worker_009"
```

`release_id_override`: giữ `None` cho `AIOU26_release` hiện tại vì manifest đã có `completed_at` và auto-resolve được `canonical_release_v001`. Chỉ đặt chuỗi release cụ thể khi cả team chủ động pin cùng một snapshot.

`hf_release_repo = None` và `hf_checkpoint_repo = None` nghĩa là dùng default trong `configs/storage.yaml`:

```text
release repo:    1thesudden/AIOU26_release
checkpoint repo: 1thesudden/AIOU26_checkpoints
```

Notebook 01 không đọc trực tiếp raw repo. Raw repo của Notebook 00B là `1thesudden/AIOU26_raw`.

Nếu chỉ chạy thử với repo test, chỉ đặt `hf_release_repo = "1thesudden/AIOU26_release_test"` khi repo đó đã tồn tại và token có quyền đọc/ghi. Nếu không chắc, giữ `None` để dùng `1thesudden/AIOU26_release`.

Mỗi teammate phải cấu hình secret trước khi chạy:

```text
HF_TOKEN hoặc AIC_HF_TOKEN
```

`HF_TOKEN` phải có quyền ghi vào Hugging Face release repo và checkpoint repo, vì Notebook 01 chạy với `--sync` để upload kết quả Phase01.

Model local/free trong Notebook 01:

```text
OCR:          5CD-AI/Vintern-1B-v3_5
ASR:          nvidia/parakeet-ctc-0.6b-vi (NeMo/FastConformer)
Shot caption: Qwen/Qwen2.5-VL-7B-Instruct 4-bit NF4
Scene boundary/summary: cùng Qwen đã load trong runtime chunk
Semantic fallback: 5CD-AI/Vintern-3B-R-beta chạy local
```

Free/local nghĩa là model chạy trên GPU của laptop, Colab free hoặc Kaggle free; pipeline không gọi semantic API bên ngoài nhưng vẫn cần đủ VRAM, runtime disk và thời gian tải model. Ảnh đưa vào VLM là representative keyframe của shot; output lưu thêm `ocr.parquet` và `shot_captions.parquet` schema v3 gồm caption, objects, actions, visible text summary.

Runtime chunking được package tự động điều chỉnh theo dung lượng scratch và RAM còn trống (RAM >8 GiB: tối đa 4 video; 4–8 GiB: tối đa 2; dưới 4 GiB: 1 và heavy-model guard sẽ dừng retryable nếu cleanup vẫn không đủ). Teammate không tự chia nhỏ `batch_id`; cứ chọn batch do Notebook 00B tạo và chạy **Run All**. OCR request batch mặc định là 4, caption batch là 2 và tự giảm về 1 khi OOM; scene boundary/summary giữ batch 1. Vintern OCR unload trước khi Qwen load. Nếu semantic fallback được kích hoạt, Qwen phải unload trước khi Vintern-3B-R-beta load và fallback được giữ cố định đến hết runtime chunk.

Nếu một video fail tại `shot_captions`, checkpoint chỉ có bốn stage hoàn tất (`shots`, `keyframes`, `asr`, `ocr`) là đúng: stage lỗi không được promote và downstream chưa chạy. Chỉ coi là checkpoint corruption khi `state.json` ghi `complete` nhưng artifact/checksum tương ứng bị thiếu hoặc sai.

Trên Kaggle, notebook dùng `/kaggle/temp/aic_phase01` cho model cache/scratch để tránh quota auto-save 20GB của `/kaggle/working`. Kết quả chính vẫn được sync lên Hugging Face.

Semantic fallback không cần API key. Nếu cả Qwen và Vintern-3B-R-beta đều lỗi, stage hiện tại fail rõ ràng và không promote checkpoint.

Không sửa `github_repo_url`, `github_branch`, `repo_dir_name`, `asr_provider`, `scratch_dir_override` nếu không có lý do cụ thể.

In [ ]:
# BƯỚC 1: Detect environment, paths và secrets. Không in secret.
import os, shutil, sys
from pathlib import Path

if "google.colab" in sys.modules:
    runtime_env = "colab"
    workspace = Path("/content/aic_phase01")
elif "KAGGLE_URL_BASE" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    runtime_env = "kaggle"
    workspace = Path("/kaggle/temp/aic_phase01")
else:
    runtime_env = "local"
    workspace = Path.cwd() / ".phase01_runtime"
workspace.mkdir(parents=True, exist_ok=True)
output_root = workspace / "output"
scratch_dir = Path(scratch_dir_override).expanduser() if scratch_dir_override else workspace / "scratch"
model_cache = workspace / "model_cache"
for path in (output_root, scratch_dir, model_cache): path.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(workspace).free / (1024**3)
if runtime_env in {"colab", "kaggle"} and free_gb < 35:
    raise RuntimeError(f"Không đủ runtime disk cho model local mặc định: {free_gb:.1f} GiB trống < 35 GiB.")

def load_secret(name):
    if os.environ.get(name): return os.environ[name]
    if runtime_env == "colab":
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    if runtime_env == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            return None
    return None

for secret_name in ("HF_TOKEN", "AIC_HF_TOKEN"):
    value = load_secret(secret_name)
    if value: os.environ[secret_name] = value
if not os.environ.get("HF_TOKEN") and os.environ.get("AIC_HF_TOKEN"): os.environ["HF_TOKEN"] = os.environ["AIC_HF_TOKEN"]
if os.environ.get("HF_TOKEN"): os.environ["AIC_HF_TOKEN"] = os.environ["HF_TOKEN"]
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Cần cấu hình HF_TOKEN trong Colab/Kaggle Secrets hoặc environment.")

os.environ["HF_HOME"] = str(model_cache / "hf")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
os.environ.setdefault("HF_XET_SHARD_CACHE_SIZE_LIMIT", str(1024**3))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["AIC_DATA_ROOT"] = str(workspace / "data")
os.environ["AIC_RUNTIME_ROOT"] = str(workspace / "runtime")
os.environ["AIC_ARTIFACT_ROOT"] = str(workspace / "artifacts")
print({"environment": runtime_env, "workspace": str(workspace), "output_root": str(output_root), "scratch": str(scratch_dir)})

In [ ]:
# BƯỚC 2: Clone/update đúng branch GitHub mà không xóa thay đổi local.
import subprocess

def run_command(command, cwd=None):
    print("RUN:", " ".join(map(str, command)))
    result = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode: raise RuntimeError(f"Command failed ({result.returncode}): {command}")
    return result

def is_repo_root(path): return (path / "system1" / "src" / "system1" / "__init__.py").is_file()
repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if is_repo_root(path)), None)
if repo_root is None:
    repo_root = workspace / repo_dir_name
    if not (repo_root / ".git").exists():
        run_command(["git", "clone", "--branch", github_branch, "--single-branch", github_repo_url, str(repo_root)], cwd=workspace)
if run_command(["git", "status", "--porcelain"], cwd=repo_root).stdout.strip():
    raise RuntimeError("Repo có thay đổi local; notebook không tự reset/stash.")
run_command(["git", "fetch", "origin", github_branch], cwd=repo_root)
local_branch = subprocess.run(["git", "show-ref", "--verify", "--quiet", f"refs/heads/{github_branch}"], cwd=repo_root).returncode == 0
run_command(["git", "switch", github_branch] if local_branch else ["git", "switch", "--track", "-c", github_branch, f"origin/{github_branch}"], cwd=repo_root)
run_command(["git", "merge", "--ff-only", f"origin/{github_branch}"], cwd=repo_root)
git_sha = run_command(["git", "rev-parse", "HEAD"], cwd=repo_root).stdout.strip()
remote_sha = run_command(["git", "rev-parse", f"origin/{github_branch}"], cwd=repo_root).stdout.strip()
actual_branch = run_command(["git", "branch", "--show-current"], cwd=repo_root).stdout.strip()
dirty = bool(run_command(["git", "status", "--porcelain"], cwd=repo_root).stdout.strip())
source_identity = {"git_commit_sha": git_sha, "local_branch": actual_branch, "expected_branch": github_branch, "origin_branch_sha": remote_sha, "dirty": dirty}
print("SOURCE IDENTITY:", source_identity)
if dirty or actual_branch != github_branch or git_sha != remote_sha:
    raise RuntimeError(f"Source code stale/mismatch; notebook will not reset local work: {source_identity}")
os.environ["AIC_EXPECTED_GIT_BRANCH"] = github_branch
os.environ["AIC_REPO_ROOT"] = str(repo_root)
os.environ["AIC_REPO_PARENT"] = str(repo_root.parent)

In [ ]:
# BƯỚC 3: Cài production dependencies và xác nhận kernel import đúng source.
system1_root = repo_root / "system1"
run_command([sys.executable, "-m", "pip", "install", "-q", "-e", f"{system1_root}[phase01-production]"])
source_root = str(system1_root / "src")
sys.path = [source_root] + [item for item in sys.path if item != source_root]
for name in list(sys.modules):
    if name == "system1" or name.startswith("system1."): del sys.modules[name]
import system1
from system1.config import load_configs
import torch
actual = Path(system1.__file__).resolve()
expected = (system1_root / "src" / "system1" / "__init__.py").resolve()
if actual != expected: raise RuntimeError(f"Import source mismatch: actual={actual}, expected={expected}")
phase01_models = load_configs(system1_root / "configs")["models"]["phase01"]
semantic = phase01_models["shot_caption"]
if semantic["provider"] != "qwen_local" or [item["provider"] for item in semantic["fallbacks"]] != ["vintern_reasoning_local"]:
    raise RuntimeError(f"Semantic model contract mismatch: {semantic}")
if any(phase01_models[stage]["model_key"] != "shot_caption" for stage in ("scene_boundary", "scene_summary")):
    raise RuntimeError("Semantic stages do not share the shot_caption runtime.")
if runtime_env in {"colab", "kaggle"} and not torch.cuda.is_available():
    raise RuntimeError("Notebook 01 local VLM default cần GPU. Bật GPU runtime trước khi Run All.")
if torch.cuda.is_available():
    print({"cuda": True, "gpu": torch.cuda.get_device_name(0), "workspace_free_gb": round(shutil.disk_usage(workspace).free / (1024**3), 2)})
print("package source preflight: OK", actual)

In [ ]:
# ============================================================
# OPTIONAL REAL-PROVIDER SMOKE TEST — chạy thủ công sau BƯỚC 3
#
# Run All an toàn vì smoke bị tắt mặc định. Khi cần test một video thật:
#   1. đặt SMOKE_TEST_VIDEO_PATH tới video local/Kaggle/Colab;
#   2. tùy chọn đặt SMOKE_TEST_METADATA_PATH;
#   3. đổi RUN_REAL_PROVIDER_SMOKE thành True và chạy riêng cell này.
#
# Artifact chỉ nằm trong TemporaryDirectory và không upload lên Hugging Face.
# ============================================================

RUN_REAL_PROVIDER_SMOKE = False
SMOKE_TEST_VIDEO_PATH = None
SMOKE_TEST_METADATA_PATH = None


def run_real_provider_smoke(video_path, metadata_path=None):
    import json
    import os
    import shutil
    import tempfile
    from pathlib import Path

    import pandas as pd

    import system1
    import system1.phase01.production as production

    from system1.artifacts.package import extract_artifact_zip
    from system1.artifacts.store import ArtifactStore
    from system1.config import (
        require_phase01_production_ready,
        resolve_phase01_config,
    )
    from system1.ingest.pipeline import run_ingestion
    from system1.phase01.checkpoint import checkpoint_root
    from system1.phase01.model_artifacts import materialize_transnet_artifact


    TEST_VIDEO_PATH = Path(video_path).expanduser().resolve()
    TEST_METADATA_PATH = (
        Path(metadata_path).expanduser().resolve()
        if metadata_path is not None
        else None
    )

    if not TEST_VIDEO_PATH.is_file():
        raise FileNotFoundError(f"Không tìm thấy video test: {TEST_VIDEO_PATH}")


    # ------------------------------------------------------------
    # Local ephemeral store giả lập HF store.
    #
    # Checkpoint + release sync đều đi vào TemporaryDirectory,
    # KHÔNG upload ra Hugging Face.
    # ------------------------------------------------------------

    class EphemeralStore:
        def __init__(self, root: Path) -> None:
            root = root.resolve()
            root.mkdir(parents=True, exist_ok=True)

            self.repo_id = "ephemeral://phase01-smoke"
            self._store = ArtifactStore(root=root)

        def __getattr__(self, name):
            return getattr(self._store, name)


    package_root = Path(system1.__file__).resolve().parents[2]
    config_dir = package_root / "configs"

    original_hf_store = production._hf_store

    smoke_result = None


    with tempfile.TemporaryDirectory(
        prefix="system1_phase01_real_smoke_"
    ) as temp_name:

        temp_root = Path(temp_name)

        # --------------------------------------------------------
        # 1. Tạo input Phase00 tạm
        # --------------------------------------------------------

        input_root = temp_root / "input"
        raw_dir = input_root / "raw_videos"
        metadata_dir = input_root / "metadata"

        raw_dir.mkdir(parents=True)
        metadata_dir.mkdir(parents=True)

        video_id_hint = TEST_VIDEO_PATH.stem

        smoke_video = raw_dir / TEST_VIDEO_PATH.name

        # Không copy video lớn.
        # Chỉ tạo symbolic link trong thư mục tạm.
        os.symlink(
            TEST_VIDEO_PATH,
            smoke_video,
        )

        smoke_metadata = metadata_dir / f"{video_id_hint}.json"

        if TEST_METADATA_PATH is not None:
            metadata_source = Path(
                TEST_METADATA_PATH
            ).expanduser().resolve()

            if not metadata_source.is_file():
                raise FileNotFoundError(metadata_source)

            shutil.copy2(
                metadata_source,
                smoke_metadata,
            )
        else:
            smoke_metadata.write_text(
                json.dumps(
                    {
                        "video_id": video_id_hint,
                        "title": None,
                    },
                    ensure_ascii=False,
                    indent=2,
                ),
                encoding="utf-8",
            )

        # --------------------------------------------------------
        # 2. Chạy Phase00 local tối thiểu chỉ để tạo:
        #
        # videos.parquet
        # media_store_manifest.parquet
        # frame timeline
        #
        # Tất cả vẫn nằm trong temp.
        # --------------------------------------------------------

        output_root = temp_root / "output"

        ingestion_report = run_ingestion(
            output_root,
            source_uri=input_root,
            max_workers=1,
            pairing_policy="strict",
            frame_timeline_policy="required",
        )

        release_dir = ingestion_report.parents[1]

        videos = pd.read_parquet(
            release_dir / "tables" / "videos.parquet"
        )

        if len(videos) != 1:
            raise RuntimeError(
                f"Smoke ingestion phải có đúng 1 video, nhận được {len(videos)}"
            )

        video_id = str(videos.iloc[0]["video_id"])

        batch_id = "smoke_000"
        worker_id = "smoke_ephemeral"

        batch_path = (
            release_dir
            / "manifests"
            / f"{batch_id}.txt"
        )

        batch_path.write_text(
            video_id + "\n",
            encoding="utf-8",
        )

        # --------------------------------------------------------
        # 3. Resolve chính production config.
        #
        # Không tạo special mock config.
        # Model/provider vẫn là model thật.
        # --------------------------------------------------------

        resolved = resolve_phase01_config(
            config_dir,
            user_settings={
                "batch_id": batch_id,
                "worker_id": worker_id,
            },
            phase00_release_id=release_dir.name,
        )

        require_phase01_production_ready(resolved)

        # --------------------------------------------------------
        # 4. Materialize TransNet thật.
        #
        # Cache nằm trong temp_root nên sẽ bị xóa sau smoke.
        # Đây chỉ DOWNLOAD model artifact, không upload.
        # --------------------------------------------------------

        transnet = materialize_transnet_artifact(
            model_config=resolved.payload["models"]["shot_detection"],
            storage_config=resolved.payload["storage"]["model_artifacts"],
            cache_root=temp_root / "model_artifacts",
        )

        # --------------------------------------------------------
        # 5. Redirect TOÀN BỘ checkpoint/release store
        # sang local temp store.
        # --------------------------------------------------------

        ephemeral_store = EphemeralStore(
            temp_root / "ephemeral_remote"
        )

        production._hf_store = (
            lambda _config, **_kwargs: ephemeral_store
        )

        scratch_root = temp_root / "scratch"

        try:
            # ----------------------------------------------------
            # sync_release=True CỐ Ý.
            #
            # Ta muốn test cả code path:
            #
            # package
            # -> upload
            # -> download verify
            # -> sync checkpoint
            #
            # nhưng "upload" ở đây chỉ upload vào
            # EphemeralStore trong temp directory.
            # ----------------------------------------------------

            report_path = production.process_production_batch(
                release_dir=release_dir,
                config=resolved,
                scratch_root=scratch_root,
                transnet_artifact_dir=transnet.root,
                sync_release=True,
            )

            report = json.loads(
                report_path.read_text(
                    encoding="utf-8"
                )
            )

            # ----------------------------------------------------
            # 6. Đọc checkpoint tạm để chứng minh tất cả stage pass
            # ----------------------------------------------------

            state_relative = (
                checkpoint_root(
                    release_dir.name,
                    video_id,
                    str(
                        resolved.payload["artifact"]
                        ["checkpoint"]["root"]
                    ),
                )
                / str(
                    resolved.payload["artifact"]
                    ["checkpoint"]["state_filename"]
                )
            )

            state = ephemeral_store.read_json(
                state_relative
            )

            stage_status = {
                stage: record["status"]
                for stage, record
                in state["stages"].items()
            }

            print("\n" + "=" * 72)
            print("PHASE01 REAL-PROVIDER SMOKE RESULT")
            print("=" * 72)

            print("video_id :", video_id)
            print("release  :", release_dir.name)

            print("\nSTAGES:")

            for stage, status in stage_status.items():
                print(
                    f"  {stage:<24} {status}"
                )

            failed_stages = {
                stage: status
                for stage, status
                in stage_status.items()
                if status != "complete"
            }

            if failed_stages:
                raise RuntimeError(
                    f"Smoke chưa hoàn tất: {failed_stages}"
                )

            # ----------------------------------------------------
            # 7. Inspect package thật trước khi temp bị xóa
            # ----------------------------------------------------

            video_report = report["videos"][0]

            artifact_path = Path(
                video_report["artifact"]
            )

            extracted = extract_artifact_zip(
                artifact_path,
                temp_root / "inspect",
                expected_video_id=video_id,
                expected_artifact_type="structure",
            )

            table_names = [
                "shots",
                "keyframes",
                "asr_segments",
                "ocr",
                "shot_captions",
                "shot_transcript_links",
                "scenes",
                "scene_transcript_links",
                "scene_summaries",
            ]

            table_counts = {}

            for table_name in table_names:
                path = extracted / f"{table_name}.parquet"
                table_counts[table_name] = len(
                    pd.read_parquet(path)
                )

            print("\nTABLE COUNTS:")

            for name, count in table_counts.items():
                print(
                    f"  {name:<24} {count}"
                )

            # ----------------------------------------------------
            # 8. Hiển thị một số semantic output quan trọng
            # ----------------------------------------------------

            captions = pd.read_parquet(
                extracted / "shot_captions.parquet"
            )

            scenes = pd.read_parquet(
                extracted / "scenes.parquet"
            )

            summaries = pd.read_parquet(
                extracted / "scene_summaries.parquet"
            )

            ocr = pd.read_parquet(
                extracted / "ocr.parquet"
            )

            print("\nOCR SAMPLE:")
            display(
                ocr.head(10)
            )

            print("\nSHOT CAPTION SAMPLE:")
            display(
                captions.head(10)
            )

            print("\nSCENES:")
            display(
                scenes.head(20)
            )

            print("\nSCENE SUMMARIES:")
            display(
                summaries.head(20)
            )

            smoke_result = {
                "video_id": video_id,
                "stages": stage_status,
                "table_counts": table_counts,
                "status": "PASS",
            }

            print("\n" + "=" * 72)
            print("SMOKE TEST: PASS")
            print("Tất cả stage production đã chạy thành công.")
            print(
                "Không có checkpoint/release nào "
                "được upload lên Hugging Face."
            )
            print(
                "TemporaryDirectory sẽ bị xóa "
                "sau khi cell kết thúc."
            )
            print("=" * 72)

        except Exception:
            # Nếu fail, cố in checkpoint để biết chính xác stage.
            try:
                state = ephemeral_store.read_json(
                    state_relative
                )

                print("\nCHECKPOINT STATE AT FAILURE:")

                for stage, record in state["stages"].items():
                    print(
                        f"  {stage:<24} "
                        f"{record['status']}"
                    )

                    if record.get("error"):
                        print(
                            "    error:",
                            record["error"],
                        )

            except Exception:
                pass

            raise

        finally:
            production._hf_store = original_hf_store


    # Tại thời điểm tới đây TemporaryDirectory đã bị xóa.
    print("\nFINAL IN-MEMORY RESULT:")
    print(smoke_result)

    return smoke_result


if RUN_REAL_PROVIDER_SMOKE:
    if not SMOKE_TEST_VIDEO_PATH:
        raise ValueError(
            "Hãy đặt SMOKE_TEST_VIDEO_PATH trước khi bật smoke test."
        )
    smoke_result = run_real_provider_smoke(
        SMOKE_TEST_VIDEO_PATH,
        SMOKE_TEST_METADATA_PATH,
    )
else:
    print("REAL-PROVIDER SMOKE TEST: skipped (disabled by default)")


In [ ]:
# BƯỚC 4: Helper CLI có streaming output và error tail.
def run_cli(arguments):
    command = [sys.executable, "-m", "system1.cli", *map(str, arguments)]
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(command, cwd=system1_root, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True); lines.append(line)
    code = process.wait()
    if code:
        raise RuntimeError(f"CLI failed ({code}). Last output:\n" + "".join(lines[-120:]))
    return "".join(lines)

In [ ]:
# BƯỚC 5: Một lệnh production — auto-resolve, restore, preflight, resume, package, sync.
command = [
    "process-batch", "--batch-id", batch_id, "--worker-id", worker_id,
    "--output", str(output_root), "--scratch-dir", str(scratch_dir),
    "--require-frame-timeline", "--restore-phase00", "--validate-remote", "--sync",
]
optional = {
    "--asr-provider": asr_provider,
    "--release-id-override": release_id_override,
    "--hf-repo-id": hf_release_repo,
    "--hf-checkpoint-repo": hf_checkpoint_repo,
    "--hf-revision": hf_revision,
    "--hf-prefix": hf_prefix,
    "--checkpoint-revision": checkpoint_revision,
    "--checkpoint-prefix": checkpoint_prefix,
}
for option, value in optional.items():
    if value not in (None, ""): command.extend([option, str(value)])
run_cli(command)

In [ ]:
# BƯỚC 6: Báo cáo ngắn sau Run All.
import json
last_run = json.loads((output_root / "phase01_last_run.json").read_text())
release_root = Path(last_run["release_dir"])
resolved = json.loads((release_root / "manifests/phase01/resolved_config.json").read_text())
report_path = release_root / "manifests/worker_reports" / f"structure_{batch_id}_{worker_id}.json"
review_path = release_root / "manifests/phase01" / f"manual_review_{batch_id}_{worker_id}.json"
report = json.loads(report_path.read_text())
review = json.loads(review_path.read_text())
print({
    "release_id": resolved["runtime"]["release_id"],
    "config_hash": resolved["config_hash"],
    "counts": report.get("counts"),
    "manual_review_status": review["status"],
    "manual_review_samples": review["sample_size_actual"],
    "report": str(report_path),
})

In [ ]:
# BƯỚC 7: Verify trực tiếp output Phase01 đã hiện diện đầy đủ trên Hugging Face.
from huggingface_hub import HfApi

release_storage = resolved["storage"]["release"]
release_id = resolved["runtime"]["release_id"]
remote_root = f"{release_id}/phase01_structure"
prefix = str(release_storage.get("prefix") or "").strip("/")
scoped_root = f"{prefix}/{remote_root}" if prefix else remote_root
api = HfApi(token=os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN"))
entries = api.list_repo_tree(
    repo_id=release_storage["repo_id"],
    repo_type=release_storage.get("repo_type", "dataset"),
    revision=release_storage.get("revision", "main"),
    path_in_repo=scoped_root,
    recursive=True,
)
remote_files = {entry.path for entry in entries if getattr(entry, "path", None)}
complete_videos = [row["video_id"] for row in report.get("videos", []) if str(row.get("status", "")).startswith("complete")]
package_root = resolved["artifact"]["package"]["root"].format(
    release_id=release_id, batch_id=batch_id, video_id=""
).strip("/")
package_filename = resolved["artifact"]["package"]["filename"]
expected = {f"{prefix}/{package_root}/{package_filename.format(video_id=video_id)}" if prefix else f"{package_root}/{package_filename.format(video_id=video_id)}" for video_id in complete_videos}
expected.update({
    f"{scoped_root}/worker_reports/structure_{batch_id}_{worker_id}.json",
    f"{scoped_root}/worker_reports/errors_{batch_id}_{worker_id}.jsonl",
    f"{scoped_root}/worker_reports/manual_review_{batch_id}_{worker_id}.json",
})
missing = sorted(expected - remote_files)
if missing: raise RuntimeError("HF Phase01 output verification failed; missing: " + ", ".join(missing))
print({
    "hf_repo": release_storage["repo_id"],
    "remote_root": scoped_root,
    "verified_files": len(expected),
    "verified_packages": len(complete_videos),
    "status": "HF phase01_structure verification: OK",
})
